In [ ]:
from os import path

dataset_path = path.join("../../", "00-input")
clean_dataset_path = path.join("../../", "01-refinement/clean")
pd_path = path.join(dataset_path, "PD - Demographic+Clinical - datasetV1.csv")
controls_path = path.join(dataset_path, "CONTROLS - Demographic+Clinical - datasetV1.csv")

whitelist_path = "data_cleaning.whitelist.json"
types_path = "data_cleaning.types.json"

In [ ]:
import os

def extract_subject_id(filename):
    return os.path.basename(filename).split("_")[0]

In [ ]:
import pandas as pd
import json

with open(whitelist_path, "r") as f:
    whitelist = json.load(f)

with open(types_path, "r") as f:
    types = json.load(f)

In [ ]:
def clean_dataset(input_path, dataset_name, output_name):
    
    df = pd.read_csv(
        input_path,
        sep = ';',                 # separatore corretto
        header = 2,                # salta le prime 2 righe
        engine = 'python',         # necessario per multilinea
        quotechar = '"',           # gestisce testi lunghi
        on_bad_lines = 'skip'      # evita crash
    )

    df.columns = df.columns.str.strip()

    rename_map = {
        k.strip(): v
        for k, v in whitelist[dataset_name].items()
        if k.strip() in df.columns
    }
    df.rename(columns=rename_map, inplace=True)

    
    valid_cols = list(rename_map.values())
    df = df[valid_cols]


    for col, t in types[dataset_name].items():
        if col in df.columns:
            try:
                if t in ["float", "int"]:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
                else:
                    df[col] = df[col].astype(t)
            except Exception as e:
                print(f"Warning: {col} -> {t}: {e}")

    if "research_time" in df.columns:
        df["research_time"] = pd.to_datetime(
            df["research_time"],
            dayfirst=True,
            errors="coerce"
        )

        df["research_time"] = df["research_time"].dt.strftime("%Y-%m-%dT%H:%M:%S")
        df["research_time"] = df["research_time"].where(df["research_time"].notnull(), None)

    
    
    if "pt_ot_status" in df.columns:
        df["pt_ot_status"] = (
            df["pt_ot_status"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    if "frequency_pt_ot" in df.columns:
        df["frequency_pt_ot"] = (
            df["frequency_pt_ot"]
            .astype(str)
            .str.lower()
            .str.strip()
        )

    df.replace({"": None, "nan": None, "none": None, "-": None}, inplace=True)

    df = df.applymap(lambda x: x.replace("\n", " ").strip() if isinstance(x, str) else x)

    display(df.head())

    output_file = path.join(clean_dataset_path, output_name)
    df.to_csv(output_file, index=False)

In [ ]:
def clean_patient_files(input_folder, dataset_name, output_name):

    all_data = []

    for file in os.listdir(input_folder):
        if file.endswith(".csv"):
            file_path = os.path.join(input_folder, file)

            subject_id = extract_subject_id(file)

            df = pd.read_csv(
                file_path,
                sep=',',
                header=0,
                engine='python',
                quotechar='"',
                on_bad_lines='skip'
            )

            df.columns = df.columns.str.strip()
            df = df.head(3)

            df["patient.subjectid"] = subject_id

            rename_map = {
                k.strip(): v
                for k, v in whitelist[dataset_name].items()
                if k.strip() in df.columns
            }

            df.rename(columns=rename_map, inplace=True)

            valid_cols = list(rename_map.values())

            if "patient.subjectid" not in valid_cols:
                valid_cols.append("patient.subjectid")

            df = df[valid_cols]

            for col, t in types[dataset_name].items():
                if col in df.columns:
                    try:
                        if t in ["float", "int"]:
                            df[col] = pd.to_numeric(df[col], errors="coerce")
                        else:
                            df[col] = df[col].astype(t)
                    except Exception as e:
                        print(f"Warning: {col} -> {t}: {e}")


            df.replace({"": None, "nan": None, "none": None, "-": None}, inplace=True)
            df = df.applymap(lambda x: x.replace("\n", " ").strip() if isinstance(x, str) else x)

            all_data.append(df)

    if all_data:
        final_df = pd.concat(all_data, ignore_index=True)
    else:
        print("No files found!")
        return

    display(final_df.head())

    output_file = path.join(clean_dataset_path, output_name)
    final_df.to_csv(output_file, index=False)

In [ ]:
clean_dataset(pd_path, "pd_first_overview", "pd_first_overview.clean.csv")

clean_dataset(controls_path, "controls_first_overview", "controls_first_overview.clean.csv")

balance_folder = path.join(dataset_path, "balance_files")

clean_patient_files(
    balance_folder,
    "balance",
    "balance.clean.csv"
)